In [1]:
# Jupyter cell: Validate a DataFrame with columns ["ID", "Phenotype"], loaded and saved via pickle

# ------------------- Config -------------------
INPUT_PICKLE = "/work/gr-fe/bryan/data/YEAST/01_raw/phenotype.pkl"       # Path to a pickled pandas DataFrame
OUTPUT_PICKLE = "/work/gr-fe/bryan/data/YEAST/02_processed/phenotype.processed.pkl"                     # If None, auto-generate "<input>.cleaned.pkl"
SAVE_CLEANED = True                      # Save cleaned DataFrame to pickle
STRICT_TWO_COLS = False                  # If True, fail when extra columns exist
EXPECTED_ID = "ID"                       # Expected ID column name (case-insensitive)
EXPECTED_PHENOTYPE = "Phenotype"         # Expected Phenotype column name (case-insensitive)

# ------------------- Imports -------------------
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------- Load -------------------
obj = pd.read_pickle(INPUT_PICKLE)['meta']
if not isinstance(obj, pd.DataFrame):
    raise TypeError(f"Pickle did not contain a pandas DataFrame (got {type(obj)}).")

df_raw = obj.copy()
print("Loaded DataFrame shape:", df_raw.shape)
display(df_raw.head(3))

# ------------------- Validation / Cleaning -------------------
def validate_id_phenotype(
    df: pd.DataFrame,
    id_col=EXPECTED_ID,
    pheno_col=EXPECTED_PHENOTYPE,
    strict_two_cols: bool = False,
):
    report = {"ok": True, "messages": []}
    df = df.copy()

    # 1) Find required columns (case-insensitive exact match)
    lower_map = {c.lower(): c for c in df.columns}
    need = {id_col.lower(): id_col, pheno_col.lower(): pheno_col}
    missing = [need[k] for k in need if k not in lower_map]
    if missing:
        report["ok"] = False
        report["messages"].append(f"Missing required columns: {missing}. Columns found: {list(df.columns)}")
        return report, df

    # 2) Rename to canonical names
    df = df.rename(columns={
        lower_map[id_col.lower()]: "ID",
        lower_map[pheno_col.lower()]: "Phenotype"
    })

    # 3) Optional: enforce exactly two columns
    if strict_two_cols and set(df.columns) != {"ID", "Phenotype"}:
        report["ok"] = False
        report["messages"].append(f"Expected exactly two columns ['ID','Phenotype'], got: {list(df.columns)}")
        return report, df

    # 4) Keep only relevant columns for cleaned output
    core = df[["ID", "Phenotype"]].copy()

    # 5) Normalize: strip whitespace and stray quotes
    def _clean_str(s):
        if pd.isna(s):
            return np.nan
        s = str(s).strip()
        s = s.strip('"').strip("'")
        return s.strip()

    core["ID"] = core["ID"].map(_clean_str)
    core["Phenotype"] = core["Phenotype"].map(_clean_str)

    # 6) Basic checks
    n_rows = len(core)
    n_missing_id = core["ID"].isna().sum()
    n_missing_ph = core["Phenotype"].isna().sum()

    if n_missing_id > 0:
        report["ok"] = False
        report["messages"].append(f"Found {n_missing_id} rows with missing ID.")
    if n_missing_ph > 0:
        report["ok"] = False
        report["messages"].append(f"Found {n_missing_ph} rows with missing Phenotype.")

    # 7) Duplicates
    dup_pair = core.duplicated(["ID", "Phenotype"]).sum()
    if dup_pair > 0:
        report["messages"].append(f"Found {dup_pair} duplicate (ID, Phenotype) pairs. These will be dropped in cleaned output.")

    # 8) Summary stats
    n_ids = core["ID"].nunique(dropna=True)
    n_ph = core["Phenotype"].nunique(dropna=True)
    report["messages"].append(f"Rows: {n_rows}, unique IDs: {n_ids}, unique Phenotypes: {n_ph}")

    # 9) Create cleaned version (drop duplicate pairs; keep one)
    core_clean = core.drop_duplicates(["ID", "Phenotype"]).reset_index(drop=True)

    return report, core_clean

report, df_clean = validate_id_phenotype(
    df_raw,
    id_col=EXPECTED_ID,
    pheno_col=EXPECTED_PHENOTYPE,
    strict_two_cols=STRICT_TWO_COLS
)

print("OK:", report["ok"])
for m in report["messages"]:
    print("-", m)

print("Cleaned shape:", df_clean.shape)
display(df_clean.head(5))

# ------------------- Save (pickle) -------------------
if SAVE_CLEANED:
    out_path = OUTPUT_PICKLE
    if out_path is None:
        p = Path(INPUT_PICKLE)
        out_path = p.with_name(p.stem + ".processed.pkl")
    df_clean.to_pickle(out_path)
    print(f"Cleaned DataFrame saved to: {out_path}")


Loaded DataFrame shape: (10241, 2)


,ID,Phenotype
0,000UH,0
1,02KOH,0
2,02KOH,1


OK: True
- Rows: 10241, unique IDs: 2417, unique Phenotypes: 14
Cleaned shape: (10241, 2)


,ID,Phenotype
0,000UH,0
1,02KOH,0
2,02KOH,1
3,03P0J,0
4,03P0J,11


Cleaned DataFrame saved to: /work/gr-fe/bryan/data/YEAST/02_processed/phenotype.processed.pkl
